In [0]:
silver_df = spark.read.table("taxi_silver.trips")

In [0]:
from pyspark.sql.functions import *

dim_date = (
    silver_df
    .select(to_date("pickup_due_ts").alias("full_date"))
    .distinct()
    .withColumn(
        "date_key",
        date_format("full_date", "yyyyMMdd")
    )
    .withColumn("day", dayofmonth("full_date"))
    .withColumn("day_name", date_format("full_date", "EEEE"))
    .withColumn("week", weekofyear("full_date"))
    .withColumn("month", month("full_date"))
    .withColumn("month_name", date_format("full_date", "MMMM"))
    .withColumn("quarter", quarter("full_date"))
    .withColumn("year", year("full_date"))
    .withColumn("weekend_flag", dayofweek("full_date").isin(1, 7))
)

In [0]:
dim_date.write.mode("overwrite").saveAsTable("taxi_gold.dim_date")

In [0]:
pickup_zones = (
    silver_df
    .select(
        col("Pickup Zone").alias("zone_code"),
        col("Pickup Lat").alias("latitude"),
        col("Pickup Long").alias("longitude")
    )
)

In [0]:
destination_zones = (
    silver_df
    .select(
        col("Destination Zone").alias("zone_code"),
        col("Destination Lat").alias("latitude"),
        col("Destination Long").alias("longitude")
    )
)

In [0]:
dim_zone = (
    pickup_zones
    .union(destination_zones)
    .dropDuplicates(["zone_code"])
)

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

dim_zone = dim_zone.withColumn(
    "zone_key",
    monotonically_increasing_id()
)

# Write then read back to materialize keys before joining
dim_zone.write.mode("overwrite").saveAsTable("taxi_gold.dim_zone")
dim_zone = spark.read.table("taxi_gold.dim_zone")

In [0]:
fact_trip = silver_df.join(
    dim_date,
    to_date(silver_df.pickup_due_ts) == dim_date.full_date,
    "left"
)

In [0]:
pickup_dim = dim_zone.select(
    col("zone_key").alias("pickup_zone_key"),
    col("zone_code").alias("pickup_zone")
)

In [0]:
fact_trip = fact_trip.join(
    pickup_dim,
    fact_trip["Pickup Zone"] == pickup_dim["pickup_zone"],
    "left"
)

In [0]:
destination_dim = dim_zone.select(
    col("zone_key").alias("destination_zone_key"),
    col("zone_code").alias("destination_zone")
)

In [0]:
fact_trip = fact_trip.join(
    destination_dim,
    fact_trip["Destination Zone"]
    == destination_dim["destination_zone"],
    "left"
)

In [0]:
fact_trip = fact_trip.select(

    monotonically_increasing_id().alias("trip_key"),
    "Booking ID",

    "date_key",

    "pickup_zone_key",
    "destination_zone_key",

    "Status",
    "Payment Type",
    "Booking Source",

    col("Driver").alias("driver_id"),
    "Vehicle",
    "Booked By",

    "Priority",

    col("Price").alias("fare_amount"),
    "Distance",

    "wait_time_minutes",
    "dispatch_to_arrival_minutes",
    "trip_duration_minutes"
)

In [0]:
fact_trip.write.format("delta").mode("overwrite").saveAsTable(
    "taxi_gold.fact_trip"
)